# Lesson 4 · Pandas in the Real World: Cleaning, Missing Values, Dates & Transformations
### Becoming a data detective: 80% of real data science is here

| | |
|---|---|
| **Type** | LESSON 04 |
| **Time** | about 150 minutes |
| **Level** | Beginner → Intermediate |
| **Prerequisites** | Lesson 3 (selecting, filtering, importing) |

**By the end of this notebook you will be able to:**

- Audit a dataset against the **seven data-quality dimensions**
- Find and remove duplicates; standardise messy text categories
- Detect impossible values, unit mix-ups and decimal-slip typos, and decide what to do with them
- Handle missing values with at least four strategies and *measure* which is best
- Parse dates, extract date parts, resample, roll and shift time series; handle time zones
- Transform columns with `map`, `apply`, `np.select`, `pd.cut` and package everything in a reusable `clean()` function

## 🛠️ Setup
Run this cell first. It imports the tools and loads the data every problem below relies on.

In [1]:

import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check, first_look

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
raw = pd.read_csv(DATA_DIR / "market_prices_raw.csv")        # the MESSY file
clean = pd.read_csv(DATA_DIR / "market_prices_clean.csv")    # the tidy twin (our "answer key" for experiments)
print("Setup complete ✔  raw:", raw.shape, "| clean:", clean.shape)


Setup complete ✔  raw: (15093, 9) | clean: (15048, 9)


## 👋 "Garbage in, garbage out"

Let me tell you a secret that no one puts in the brochure: **data scientists spend most of their time cleaning data.**
It isn't glamorous, but it is where careers are made, because a model trained on a column where *one maize price was typed
as 4,800 instead of 48* will confidently predict nonsense.

Today you work with `market_prices_raw.csv`: the same weekly prices you've seen, but photographed the way real data actually arrive:
with typos, gaps, duplicates and inconsistent labels. Think of yourself as a detective. **Every column is a witness. Some are lying.**

### The seven questions of data quality

| Dimension | The question | Typical test |
|---|---|---|
| **Completeness** | Are values missing? | `df.isna().sum()` |
| **Uniqueness** | Are rows repeated? | `df.duplicated().sum()` |
| **Validity** | Are values in the allowed range/format? | boolean rules (`price > 0`) |
| **Consistency** | Is the same thing always written the same way? | `value_counts()` on categories |
| **Plausibility** | Are values realistic? | `describe()`, outlier rules, domain knowledge |
| **Continuity / timeliness** | Are dates complete and in range? | `min/max`, `diff()` |
| **Integrity** | Do keys match across tables? | merges (Lesson 5) |

We'll tick these off one by one.

## 1 · The audit: look before you touch

In [2]:
first_look(raw)

📐 Shape: 15,093 rows × 9 columns
🔁 Fully duplicated rows: 45


,dtype,missing,missing_%,unique
record_id,str,0,0.0,15048
date,str,0,0.0,209
recorded_at,str,0,0.0,13401
crop,str,0,0.0,8
market,str,0,0.0,10
county,str,0,0.0,46
price_kes_per_kg,float64,353,2.3,1742
price_unit,str,0,0.0,2
volume_kg,float64,407,2.7,4205


,record_id,date,recorded_at,crop,market,county,price_kes_per_kg,price_unit,volume_kg
0,R012281,2024-04-08,2024-04-08 07:51:00,Cabbage,Machakos Market,Machakos,40.8,KES/kg,10520.0
1,R008502,2023-04-10,2023-04-10 07:46:00,Potatoes,City Market,Nairobi,64.3,KES/kg,19790.0
2,R014632,2024-11-25,2024-11-25 09:35:00,Beans,Karatina Market,Nyeri,168.1,KES/kg,14540.0
3,R014555,2024-11-18,2024-11-18 07:55:00,Maize,Eldoret Municipal Market,Uasin Gishu,50.9,KES/kg,34110.0
4,R012442,2024-04-22,2024-04-22 09:13:00,Tomatoes,Nakuru Municipal Market,Nakuru,90.5,KES/kg,21370.0


**Write down what you notice** before reading on (seriously; pause and list at least three things):

* `price_kes_per_kg` and `volume_kg` have gaps → *completeness*.
* There are fully duplicated rows → *uniqueness*.
* `date` and `recorded_at` are **text** (dtype `object`/`str`), not dates → we can't do date maths yet.
* `price_unit` has more than one value?! → *consistency / validity*.
* `county` has far more distinct labels than the 9 counties our markets sit in (`nairobi`, `NAIROBI`, `' Nairobi'`…) → *consistency*.

In [3]:
print(raw["price_unit"].value_counts(), "\n")
print("Distinct county labels:", raw["county"].nunique())
print(raw["county"].value_counts().to_string())

price_unit
KES/kg          15033
KES/90kg bag       60
Name: count, dtype: int64 

Distinct county labels: 46
county
Nairobi         2832
Mombasa         1614
Nyeri           1611
Kisumu          1611
Nakuru          1608
Uasin Gishu     1586
Machakos        1212
Meru            1212
Kiambu          1196
 Nairobi          32
Nairobi           29
nairobi           27
Uasin Gishu       27
 Uasin Gishu      25
NAIROBI           23
NYERI             22
Mombasa           21
 Nakuru           19
 Nyeri            18
NAKURU            18
meru              18
Kisumu            18
kisumu            18
kiambu            17
nakuru            17
KIAMBU            16
Machakos          15
 Kisumu           15
 Kiambu           15
UASIN GISHU       15
KISUMU            15
Nakuru            14
Nyeri             14
 Mombasa          14
 Machakos         13
Uasin-Gishu       13
MOMBASA           13
Meru              12
Kiambu            12
mombasa           12
nyeri             11
machakos          10
M

## 2 · Uniqueness: duplicates

`duplicated()` marks the *second and later* copies of a row. `keep=False` marks *all* copies so you can see them side by side.

In [4]:
print("exact duplicate rows:", raw.duplicated().sum())
print("duplicated record_ids:", raw["record_id"].duplicated().sum(), " <- record_id should be unique, so this agrees")
raw[raw.duplicated(keep=False)].sort_values("record_id").head(6)

exact duplicate rows: 45
duplicated record_ids: 45  <- record_id should be unique, so this agrees


,record_id,date,recorded_at,crop,market,county,price_kes_per_kg,price_unit,volume_kg
164,R000649,2021-03-08,2021-03-08 09:34:00,Beans,City Market,Nairobi,159.1,KES/kg,10270.0
6648,R000649,2021-03-08,2021-03-08 09:34:00,Beans,City Market,Nairobi,159.1,KES/kg,10270.0
8007,R000683,2021-03-08,2021-03-08 08:29:00,Maize,Kongowea Market,Mombasa,49.3,KES/kg,55670.0
521,R000683,2021-03-08,2021-03-08 08:29:00,Maize,Kongowea Market,Mombasa,49.3,KES/kg,55670.0
347,R000719,2021-03-08,2021-03-08 07:54:00,Potatoes,Wakulima Market,Nairobi,44.4,KES/kg,35030.0
721,R000719,2021-03-08,2021-03-08 07:54:00,Potatoes,Wakulima Market,Nairobi,44.4,KES/kg,35030.0


In [5]:
df = raw.drop_duplicates(subset="record_id").copy()        # keep the first copy of each record
print("rows before:", len(raw), "| after:", len(df))

rows before: 15093 | after: 15048


> 📝 **Note.** Two kinds of duplicate: **exact** (every column the same) and **key** duplicates (same ID or same *natural key* such as market+crop+date, but different values). Exact ones are safe to drop. Key duplicates need a decision, which version is right?

## 3 · Consistency: one fact, one spelling

Counties come in as `Nairobi`, `nairobi`, `NAIROBI`, `' Nairobi'`, `Uasin-Gishu`… A `groupby` would treat each as a *different* county, quietly splitting your statistics.

The text toolbox is the `.str` accessor:

In [6]:
s = pd.Series([" Nairobi ", "nairobi", "UASIN GISHU", "Uasin-Gishu"])
print(s.str.strip().str.replace("-", " ", regex=False).str.title().tolist())

['Nairobi', 'Nairobi', 'Uasin Gishu', 'Uasin Gishu']


In [7]:
df["county"] = (df["county"]
                .str.strip()                          # remove leading/trailing spaces
                .str.replace("-", " ", regex=False)   # hyphen -> space
                .str.title())                          # consistent capitalisation
print(df["county"].value_counts().to_string())
assert df["county"].nunique() == 9

county
Nairobi        2926
Nyeri          1672
Uasin Gishu    1672
Nakuru         1672
Mombasa        1672
Kisumu         1672
Machakos       1254
Kiambu         1254
Meru           1254


A safer pattern when categories are a *known list*: check against it.

In [8]:
valid_counties = {"Nairobi", "Mombasa", "Kisumu", "Nakuru", "Uasin Gishu", "Meru", "Kiambu", "Nyeri", "Machakos"}
unknown = set(df["county"].unique()) - valid_counties
print("Unknown labels:", unknown or "none ✔")

Unknown labels: none ✔


> 💡 **Intuition.** Cleaning text is a *conversation* with the data: apply a rule, re-check `value_counts()`, repeat. Never assume the rule worked: **verify**.

## 4 · Validity & plausibility: values that cannot (or should not) be true

### 4.1 Impossible values
A price can't be zero or negative, and volume traded can't be negative.

In [9]:
print("price <= 0   :", (df["price_kes_per_kg"] <= 0).sum())
print("volume < 0   :", (df["volume_kg"] < 0).sum())
df[(df["price_kes_per_kg"] <= 0) | (df["volume_kg"] < 0)][["record_id", "crop", "market", "price_kes_per_kg", "volume_kg"]]

price <= 0   : 6
volume < 0   : 5


,record_id,crop,market,price_kes_per_kg,volume_kg
560,R008935,Bananas,Eldoret Municipal Market,46.0,-16470.0
840,R006900,Cabbage,Thika Market,35.6,-14530.0
1978,R013398,Potatoes,City Market,0.0,26010.0
5322,R005028,Cabbage,Thika Market,27.9,-22060.0
6757,R012932,Potatoes,Machakos Market,0.0,18810.0
6922,R000172,Onions,Kibuye Market,0.0,14830.0
7420,R013986,Kale,Karatina Market,0.0,9480.0
13862,R010520,Beans,Eldoret Municipal Market,0.0,14490.0
14136,R001030,Tomatoes,Karatina Market,62.6,-21980.0
14295,R013107,Kale,City Market,0.0,13520.0


### 4.2 The sneaky one: a unit mix-up

Earlier, `price_unit` showed a second value. Some *maize* records were reported **per 90 kg bag** (the standard maize sack in Kenya),
while the column header insists on *per kg*. Let's see how they stand out:

In [10]:
bag = df["price_unit"] == "KES/90kg bag"
print("Rows reported per bag:", bag.sum())
print(df.loc[bag, ["crop", "market"]].value_counts().to_string())
print("\nMedian maize price, per-kg rows :", df.loc[(df.crop == "Maize") & ~bag, "price_kes_per_kg"].median())
print("Median maize price, per-bag rows:", df.loc[bag, "price_kes_per_kg"].median())

Rows reported per bag: 60
crop   market                  
Maize  Nakuru Municipal Market     31
       Eldoret Municipal Market    29

Median maize price, per-kg rows : 63.0
Median maize price, per-bag rows: 5139.0


A price of ~5,000 "per kg" would be a world record. The fix is a **unit conversion**, not deletion: the information is perfectly good, just in different units:

In [11]:
df.loc[bag, "price_kes_per_kg"] = df.loc[bag, "price_kes_per_kg"] / 90
df.loc[bag, "price_unit"] = "KES/kg"
print(df["price_unit"].value_counts())

price_unit
KES/kg    15048
Name: count, dtype: int64


> ⚠️ **Common mistake.** **Order of operations matters.** If we had hunted outliers *before* fixing units, the 60 bag-prices would have been flagged as errors and thrown away. Fix systematic problems (units, spelling) first, *then* hunt random ones.

### 4.3 Decimal-slip typos: compare each value with its own "neighbourhood"

A global rule like "price > 200 is an error" fails because beans legitimately cost 3× maize. Instead, compare each price with the **typical price of the same crop in the same market**.
We use `groupby(...).transform("median")`: it returns a column the same length as the table, so we can divide row by row. (We study `groupby` fully in Lesson 5.)

In [12]:
group_median = df.groupby(["crop", "market"])["price_kes_per_kg"].transform("median")
ratio = df["price_kes_per_kg"] / group_median
print(ratio.describe().round(2))

suspects = df[(ratio > 3) | (ratio < 1 / 3)]
print("\nSuspicious rows:", len(suspects), "(price more than 3x above or below its group's median)")
suspects[["record_id", "crop", "market", "price_kes_per_kg"]].assign(typical=group_median[suspects.index].round(1)).head(8)

count    14697.00
mean         1.02
std          0.38
min          0.00
25%          0.92
50%          1.00
75%          1.09
max         13.78
Name: price_kes_per_kg, dtype: float64

Suspicious rows: 38 (price more than 3x above or below its group's median)


,record_id,crop,market,price_kes_per_kg,typical
1961,R011744,Beans,Eldoret Municipal Market,14.24,138.3
1978,R013398,Potatoes,City Market,0.00,51.1
2202,R012592,Tomatoes,Thika Market,7.09,84.0
2992,R014470,Onions,Wakulima Market,1029.00,82.8
3013,R011792,Onions,Nakuru Municipal Market,10.50,79.4
3299,R001339,Maize,Machakos Market,6.44,67.9
4335,R008046,Kale,Nakuru Municipal Market,428.00,33.8
5238,R008002,Kale,Eldoret Municipal Market,415.00,34.4


Now **decide**. For each bad value you have three options:

| Option | When it is right |
|---|---|
| **Correct** it (e.g. ÷10) | you're *certain* of the cause |
| **Set to missing** and impute later | you know it's wrong but not by how much |
| **Drop the row** | the row is useless without that value and you have plenty of data |

We choose option 2 (set to `NaN`): we never *invent* a certainty we don't have, and a flag column will record that we did it.

In [13]:
bad_price = (df["price_kes_per_kg"] <= 0) | (ratio > 3) | (ratio < 1 / 3)
df["price_was_invalid"] = bad_price
df.loc[bad_price, "price_kes_per_kg"] = np.nan
df.loc[df["volume_kg"] < 0, "volume_kg"] = np.nan
print("Prices set to missing for being invalid:", int(bad_price.sum()))
print(df["price_kes_per_kg"].describe().round(1))

Prices set to missing for being invalid: 38
count    14659.0
mean        68.9
std         40.2
min         20.5
25%         40.8
50%         53.7
75%         82.0
max        213.7
Name: price_kes_per_kg, dtype: float64


> 🌍 **In the real world.** Document every cleaning decision (a `README`, a markdown cell, a *decision log*). Six months later, you or an auditor will ask *“why does 2022 have fewer rows?”*.

## 5 · Completeness: missing values

### 5.1 Measure

In [14]:
missing = pd.DataFrame({"missing": df.isna().sum(), "percent": (df.isna().mean() * 100).round(2)})
missing[missing["missing"] > 0]

,missing,percent
price_kes_per_kg,389,2.59
volume_kg,411,2.73


### 5.2 Understand *why*: is the missingness random?

Statisticians distinguish three stories:

* **MCAR** (completely at random): gaps unrelated to anything: a dropped phone call.
* **MAR** (at random *given other columns*): gaps depend on something we *can* see: e.g. one market reports less often.
* **MNAR** (not at random): the gap depends on the missing value itself: e.g. traders hide very high prices.

Let's check for patterns by market and by crop:

In [15]:
is_missing = df["price_kes_per_kg"].isna()
print("Share of price missing, by market:")
print((is_missing.groupby(df["market"]).mean() * 100).round(1).sort_values(ascending=False).to_string())
print("\nBy crop:")
print((is_missing.groupby(df["crop"]).mean() * 100).round(1).sort_values(ascending=False).to_string())

Share of price missing, by market:
market
Kongowea Market             6.5
Thika Market                2.8
Karatina Market             2.6
Kibuye Market               2.2
Nakuru Municipal Market     2.2
Eldoret Municipal Market    2.0
Wakulima Market             1.9
Meru Municipal Market       1.8
Machakos Market             1.7
City Market                 1.6

By crop:
crop
Tomatoes    4.8
Onions      2.8
Potatoes    2.7
Beans       2.5
Cabbage     2.2
Kale        2.2
Bananas     1.9
Maize       1.8


> 🧠 **Pause and predict.** What do you conclude? (Kongowea Market and Tomatoes have visibly higher gaps → the missingness is **not** completely random. Dropping rows would under-represent Mombasa tomatoes, *biasing* any analysis of them.)

### 5.3 Strategies: and a fair test

| Strategy | Code | Good when |
|---|---|---|
| Drop rows | `df.dropna(subset=["price_kes_per_kg"])` | few gaps, missing completely at random |
| Fill with a constant/global mean | `fillna(df.price.mean())` | almost never for prices (ignores crop!) |
| Fill with **group median** | `groupby(...).transform("median")` | stable groups |
| **Interpolate in time** | `groupby(...).transform(lambda s: s.interpolate())` | time series: neighbours are informative |
| Add an **indicator flag** | `df["x_was_missing"] = df["x"].isna()` | always: it keeps the information *that* it was missing |

Because we generated this data, we know the **true** price behind every gap (it's in `clean`). In real life you can't peek, but you can
*hide* known values on purpose and test your method. Let's score three strategies:

In [16]:
work = df.copy()
work["date"] = pd.to_datetime(work["date"])
work = work.sort_values(["crop", "market", "date"])          # time order inside each series is essential for interpolation
miss = work["price_kes_per_kg"].isna()
truth = clean.set_index("record_id")["price_kes_per_kg"].loc[work.loc[miss, "record_id"]].to_numpy()

key = ["crop", "market"]
fills = {
    "global mean":   work["price_kes_per_kg"].fillna(work["price_kes_per_kg"].mean()),
    "group median":  work["price_kes_per_kg"].fillna(work.groupby(key)["price_kes_per_kg"].transform("median")),
    "interpolation": work.groupby(key)["price_kes_per_kg"].transform(lambda s: s.interpolate(limit_direction="both")),
}
print(f"{miss.sum()} gaps. Mean absolute error (KES/kg) versus the TRUE price:\n")
for name, filled in fills.items():
    mae = np.abs(filled[miss].to_numpy() - truth).mean()
    print(f"  {name:14s} MAE = {mae:5.2f}")

389 gaps. Mean absolute error (KES/kg) versus the TRUE price:

  global mean    MAE = 29.80
  group median   MAE =  7.61
  interpolation  MAE =  2.62


> 💡 **Intuition.** Interpolation wins because **yesterday's price is the best predictor of today's**. The global mean is wrong in a *predictable* way (it treats beans like maize). Choose a fill strategy that respects the structure of the data.

> 🤖 **Why machine learning cares.** Most ML libraries refuse `NaN`s. How you fill them is a *modelling choice*: it can add bias. Keeping a `was_missing` flag lets a model learn from the pattern of missingness itself.

## 6 · Dates and timestamps

Text dates can't be sorted by month, resampled or subtracted. Convert with `pd.to_datetime`.

In [17]:
print(df["date"].dtype, "->", end=" ")
df["date"] = pd.to_datetime(df["date"])
df["recorded_at"] = pd.to_datetime(df["recorded_at"])
print(df["date"].dtype)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

str -> datetime64[us]
Date range: 2021-01-04 to 2024-12-30


If a column has bad dates, `errors="coerce"` turns them into `NaT` ("not a time") instead of crashing, and `format=` speeds up parsing and removes ambiguity (is `03/04/2022` 3 April or 4 March?).

In [18]:
messy = pd.Series(["2022-03-04", "2022-13-45", "not a date"])
print(pd.to_datetime(messy, errors="coerce", format="%Y-%m-%d").tolist())

[Timestamp('2022-03-04 00:00:00'), NaT, NaT]


### 6.1 The `.dt` accessor: date parts

In [19]:
d = df["date"].dt
parts = pd.DataFrame({
    "date": df["date"], "year": d.year, "month": d.month, "month_name": d.month_name(),
    "quarter": d.quarter, "week": d.isocalendar().week, "weekday": d.day_name(),
})
parts.head(3)

,date,year,month,month_name,quarter,week,weekday
0,2024-04-08,2024,4,April,2,15,Monday
1,2023-04-10,2023,4,April,2,15,Monday
2,2024-11-25,2024,11,November,4,48,Monday


### 6.2 Date arithmetic and filtering

In [20]:
span = df["date"].max() - df["date"].min()
print("Span:", span, "=", span.days // 7, "weeks")
print("Plus 30 days:", (df["date"].min() + pd.Timedelta(days=30)).date())

drought = df[(df["date"] >= "2022-03-01") & (df["date"] < "2023-03-01")]
print("Rows in the drought year:", len(drought))

Span: 1456 days 00:00:00 = 208 weeks
Plus 30 days: 2021-02-03


Rows in the drought year: 3744


### 6.3 Time series superpowers: DatetimeIndex, resample, rolling, shift

Make the date the **index** and pandas understands time. We focus on *one* series: tomatoes at Wakulima.

In [21]:
tom = (df[(df["crop"] == "Tomatoes") & (df["market"] == "Wakulima Market")]
       .set_index("date").sort_index()["price_kes_per_kg"]
       .interpolate())
print(tom.loc["2022-03"].round(1).to_string())          # "partial string indexing": everything in March 2022

date
2022-03-07     97.9
2022-03-14     95.3
2022-03-21     89.7
2022-03-28    103.8


In [22]:
monthly = tom.resample("ME").mean()                     # ME = month end. Group by month in one call
print(monthly.head(4).round(1).to_string())

smooth = tom.rolling(window=4).mean()                   # 4-week moving average (first 3 values are NaN)
wow = tom.pct_change() * 100                            # week-on-week % change
lag1 = tom.shift(1)                                     # last week's price, aligned to this week -> a classic ML feature!

pd.DataFrame({"price": tom, "4wk_avg": smooth, "wow_%": wow, "last_week": lag1}).round(1).iloc[3:7]

date
2021-01-31    68.1
2021-02-28    71.4
2021-03-31    74.1
2021-04-30    62.4
Freq: ME


,price,4wk_avg,wow_%,last_week
date,,,,
2021-01-25,65.0,68.1,-1.4,65.9
2021-02-01,59.3,64.8,-8.8,65.0
2021-02-08,71.0,65.3,19.7,59.3
2021-02-15,78.0,68.3,9.9,71.0


> ⚠️ **Common mistake.** `rolling`, `shift`, `diff` and `pct_change` assume the rows are **in time order and evenly spaced**. Sort first (`sort_index()`), and check for gaps with `index.to_series().diff().value_counts()`.

> 🤖 **Why machine learning cares.** `shift(1)` produces *last week's price* as a column. Time-series ML models often use lags like this as input features, and, critically, you must never use *tomorrow's* data to predict today (**leakage**).

### 6.4 Time-of-day and time zones

`recorded_at` carries the time the enumerator logged the price. Timestamps without a zone are *naive*; attach the zone to make them *aware*.

In [23]:
ts = df["recorded_at"]
print("Typical logging hour:", ts.dt.hour.mode()[0], "| earliest:", ts.dt.time.min(), "| latest:", ts.dt.time.max())
aware = ts.dt.tz_localize("Africa/Nairobi")             # East Africa Time, UTC+3
print(aware.iloc[0], " ->  UTC:", aware.dt.tz_convert("UTC").iloc[0])

Typical logging hour: 8 | earliest: 06:00:00 | latest: 10:59:00
2024-04-08 07:51:00+03:00  ->  UTC: 2024-04-08 04:51:00+00:00


## 7 · Transformations: reshaping values

| Tool | Use |
|---|---|
| `map(dict)` | recode categories |
| `replace` | swap specific values |
| `apply(func)` | custom logic per value/row (**slow**: try vectorised first) |
| `np.select(conds, choices)` | multi-branch if/elif/else, vectorised |
| `pd.cut / pd.qcut` | bin numbers into ranges / equal-size groups |
| `astype("category")` | memory-efficient labels |

In [24]:
# 1. map: recode crop -> food group
group = {"Maize": "Staple", "Beans": "Staple", "Potatoes": "Staple",
         "Tomatoes": "Vegetable", "Kale": "Vegetable", "Cabbage": "Vegetable", "Onions": "Vegetable", "Bananas": "Fruit"}
df["food_group"] = df["crop"].map(group)

# 2. pd.cut: price bands with human labels
df["price_band"] = pd.cut(df["price_kes_per_kg"], bins=[0, 50, 100, np.inf], labels=["low", "medium", "high"])

# 3. np.select: multi-condition label
conditions = [df["price_kes_per_kg"].isna(), df["volume_kg"] > 40000, df["volume_kg"] < 10000]
choices = ["no price", "busy", "quiet"]
df["activity"] = np.select(conditions, choices, default="normal")

# 4. category dtype: smaller memory
before = df["market"].memory_usage(deep=True)
df["market"] = df["market"].astype("category")
print(f"market column memory: {before/1e3:.0f} KB -> {df['market'].memory_usage(deep=True)/1e3:.0f} KB")

df[["crop", "food_group", "price_kes_per_kg", "price_band", "activity"]].head(4)

market column memory: 1636 KB -> 665 KB


,crop,food_group,price_kes_per_kg,price_band,activity
0,Cabbage,Vegetable,40.8,low,normal
1,Potatoes,Staple,64.3,medium,normal
2,Beans,Staple,168.1,high,normal
3,Maize,Staple,50.9,medium,normal


> 📝 **Note.** `apply` with a Python function runs row by row (slow). For 15,000 rows it's fine; for 15 million it's painful. Ask first: *is there a vectorised way?* (`np.where`, `np.select`, `.str`, `.dt`, arithmetic). Use `apply` as the last resort.

### 7.1 Put it all in one reusable function

Cleaning done by hand in 30 notebook cells can't be reproduced. A **function** can: on next month's file too. This is the pipeline we just built:

In [25]:
def clean_prices(raw):
    """Turn the raw market-price export into an analysis-ready table. Every step is documented."""
    d = raw.drop_duplicates(subset="record_id").copy()                                    # 1. uniqueness
    d["county"] = d["county"].str.strip().str.replace("-", " ", regex=False).str.title()  # 2. consistency
    d["date"] = pd.to_datetime(d["date"])                                                 # 3. types
    d["recorded_at"] = pd.to_datetime(d["recorded_at"])

    bag = d["price_unit"] == "KES/90kg bag"                                               # 4. units FIRST
    d.loc[bag, "price_kes_per_kg"] = d.loc[bag, "price_kes_per_kg"] / 90
    d.loc[bag, "price_unit"] = "KES/kg"

    key = ["crop", "market"]
    ratio = d["price_kes_per_kg"] / d.groupby(key)["price_kes_per_kg"].transform("median")   # 5. validity
    bad = (d["price_kes_per_kg"] <= 0) | (ratio > 3) | (ratio < 1 / 3)
    d.loc[bad, "price_kes_per_kg"] = np.nan
    d.loc[d["volume_kg"] < 0, "volume_kg"] = np.nan

    d["price_imputed"] = d["price_kes_per_kg"].isna()                                     # 6. completeness (+ flag)
    d["volume_imputed"] = d["volume_kg"].isna()
    d = d.sort_values(key + ["date"])
    d["price_kes_per_kg"] = d.groupby(key)["price_kes_per_kg"].transform(lambda s: s.interpolate(limit_direction="both"))
    d["volume_kg"] = d["volume_kg"].fillna(d.groupby(key)["volume_kg"].transform("median"))
    return d.reset_index(drop=True)


tidy = clean_prices(raw)
print("rows:", len(tidy), "| missing prices:", tidy["price_kes_per_kg"].isna().sum(), "| counties:", tidy["county"].nunique())
print("price range:", tidy["price_kes_per_kg"].min(), "-", tidy["price_kes_per_kg"].max())
print("Prices imputed:", int(tidy["price_imputed"].sum()), "(", round(tidy["price_imputed"].mean() * 100, 1), "%)")

rows: 15048 | missing prices: 0 | counties: 9
price range: 20.5 - 213.7
Prices imputed: 389 ( 2.6 %)


**Quality gate.** End every cleaning step with *assertions*: tiny automatic tests that shout if something is wrong:

In [26]:
assert tidy["record_id"].is_unique
assert tidy["county"].nunique() == 9
assert tidy["price_kes_per_kg"].between(10, 300).all()
assert (tidy["volume_kg"] > 0).all()
assert tidy["price_unit"].eq("KES/kg").all()
print("All quality gates passed ✔")

All quality gates passed ✔


And since we hold the truth, one last honesty check: how close is the cleaned table to the real prices?

In [27]:
cmp = tidy.merge(clean[["record_id", "price_kes_per_kg"]], on="record_id", suffixes=("", "_true"))
err = (cmp["price_kes_per_kg"] - cmp["price_kes_per_kg_true"]).abs()
print(f"Overall mean error: {err.mean():.3f} KES/kg   | on untouched rows: {err[~cmp['price_imputed']].mean():.3f}   | on repaired rows: {err[cmp['price_imputed']].mean():.2f}")

Overall mean error: 0.068 KES/kg   | on untouched rows: 0.000   | on repaired rows: 2.62

---
# 🧑‍🏫 Worked examples

## Worked example 1: "How many *real* counties are in the raw file?"

> **Thinking steps.** (1) Count distinct raw labels → inflated. (2) Apply the three text rules. (3) Count again. (4) Verify against the expected list.

In [28]:
labels = raw["county"]
fixed = labels.str.strip().str.replace("-", " ", regex=False).str.title()
print("raw distinct:", labels.nunique(), "-> after cleaning:", fixed.nunique(), "|", sorted(fixed.unique()))

raw distinct: 46 -> after cleaning: 9 | ['Kiambu', 'Kisumu', 'Machakos', 'Meru', 'Mombasa', 'Nairobi', 'Nakuru', 'Nyeri', 'Uasin Gishu']


## Worked example 2: "Which quarter had the highest average maize price?"

> **Thinking steps.** (1) Work on `clean` (already tidy). (2) Parse dates. (3) Create `year` and `quarter`. (4) Group and compare.

In [29]:
m = clean[clean["crop"] == "Maize"].copy()
m["date"] = pd.to_datetime(m["date"])
q = m.groupby([m["date"].dt.year.rename("year"), m["date"].dt.quarter.rename("quarter")])["price_kes_per_kg"].mean().round(1)
print(q.sort_values(ascending=False).head(3))

year  quarter
2022  3          76.6
2023  1          75.8
      2          75.6
Name: price_kes_per_kg, dtype: float64


---
# 🏋️ Problems

Variables from Setup: `raw` (messy) and `clean` (tidy). Each problem starts from one of them.

### 🧩 Problem 4.1: Rows left after removing duplicates

Remove exact duplicate rows from `raw`. How many rows remain?

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

`raw.duplicated()` marks repeated rows; `drop_duplicates()` removes them.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`len(...)` of the result.

</details>

*Auto-check id:* `04-p1`  ·  *Set the variable `answer`, then run the cell.*

In [30]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p1', answer)

⏳ 04-p1: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.2: How many distinct counties after cleaning?

Clean the `county` column of `raw` by: stripping spaces, replacing hyphens `-` with spaces, and applying **title case**. How many distinct counties are there afterwards?

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

Chain `.str.strip()`, `.str.replace(...)`, `.str.title()`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Use `regex=False` in `replace` when the pattern is a plain character, then `.nunique()`.

</details>

*Auto-check id:* `04-p2`  ·  *Set the variable `answer`, then run the cell.*

In [31]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p2', answer)

⏳ 04-p2: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.3: Eldoret maize, in the right units

Starting from `raw` **after dropping duplicates by `record_id`**: rows with `price_unit == 'KES/90kg bag'` must be converted to KES per kg (divide the price by 90). Then compute the **median** `price_kes_per_kg` of **Maize** at **Eldoret Municipal Market** (NaNs are ignored automatically). Round to 2 decimals.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

`d = raw.drop_duplicates(subset='record_id').copy()`: the `.copy()` avoids accidental edits to `raw`.

</details>

<details>
<summary>💡 Hint 2</summary>

`bag = d['price_unit'] == 'KES/90kg bag'` then `d.loc[bag, 'price_kes_per_kg'] /= 90`.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

Filter maize + Eldoret, take `.median()`, round.

</details>

*Auto-check id:* `04-p3`  ·  *Set the variable `answer`, then run the cell.*

In [32]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p3', answer)

⏳ 04-p3: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.4: How much price data is missing?

After dropping duplicates by `record_id`, what **percentage** of `price_kes_per_kg` values are missing? Round to 2 decimals (e.g. `2.35` for 2.35%).

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

`isna()` gives True for missing values; the **mean** of Booleans is a proportion.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Multiply by 100 and round.

</details>

*Auto-check id:* `04-p4`  ·  *Set the variable `answer`, then run the cell.*

In [33]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p4', answer)

⏳ 04-p4: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.5: Impossible values

After dropping duplicates by `record_id`, count the rows that have **a price ≤ 0 OR a negative volume** (a row meeting both conditions counts once).

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

Two Boolean conditions joined with `|` (parentheses!).

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

NaN comparisons are False, so missing values don't count: that's what we want.

</details>

*Auto-check id:* `04-p5`  ·  *Set the variable `answer`, then run the cell.*

In [34]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p5', answer)

⏳ 04-p5: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.6: Fill tomato gaps with a group median

From `raw` (after dropping duplicates by `record_id`), keep only **Tomatoes** rows. Fill missing `price_kes_per_kg` with the **median price of the same market**. (Ignore other issues such as typos.) Return the **mean** of the filled price column, rounded to 2 decimals.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

`groupby('market')['price_kes_per_kg'].transform('median')` returns a median for every row, aligned with the table.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`s.fillna(medians)` fills each gap with the matching median. Then `.mean()`.

</details>

*Auto-check id:* `04-p6`  ·  *Set the variable `answer`, then run the cell.*

In [35]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p6', answer)

⏳ 04-p6: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.7: Records in Q4 2023

Using `raw` after dropping duplicates by `record_id`: parse `date` as datetime and count the records dated in **Q4 2023** (October–December 2023).

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

`pd.to_datetime(d['date'])` then the `.dt` accessor has `.year` and `.quarter`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`((dt.year == 2023) & (dt.quarter == 4)).sum()`

</details>

*Auto-check id:* `04-p7`  ·  *Set the variable `answer`, then run the cell.*

In [36]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p7', answer)

⏳ 04-p7: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.8: Peak month at Nakuru

Using the tidy `clean` table: parse `date`, select **Maize** at **Nakuru Municipal Market**, set the date as the index, and **resample by month-end** (`'ME'`) taking the mean price. Return the month with the highest average as a string like `'2023-02'`.

*Expected type:* string `YYYY-MM`.

<details>
<summary>💡 Hint 1</summary>

Filter, then `.set_index('date')['price_kes_per_kg']`.

</details>

<details>
<summary>💡 Hint 2</summary>

`.resample('ME').mean()` gives monthly means; `.idxmax()` gives the winning timestamp.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`.strftime('%Y-%m')` formats a Timestamp as text.

</details>

*Auto-check id:* `04-p8`  ·  *Set the variable `answer`, then run the cell.*

In [37]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p8', answer)

⏳ 04-p8: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 4.9: How many 'high' price records?

On `clean`, create price bands with `pd.cut`: **low** (0–50], **medium** (50–100], **high** (100–∞). How many rows fall in the **high** band?

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

`pd.cut(series, bins=[0, 50, 100, np.inf], labels=['low','medium','high'])`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Compare the result with `'high'` and sum, or use `.value_counts()`.

</details>

*Auto-check id:* `04-p9`  ·  *Set the variable `answer`, then run the cell.*

In [38]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('04-p9', answer)

⏳ 04-p9: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### ✍️ Problem 4.10: Your decision log

Two analysts disagree about the **decimal-slip prices** (e.g. a 60 KES/kg tomato recorded as 600). Analyst A wants to *drop those rows*; Analyst B wants to *divide by 10* to correct them. In 4–6 sentences, argue which approach you would choose **and** what you would write in the decision log. Consider: how sure are you of the cause? how many rows? what would a wrong correction cost?

✍️ *Write your answer here (double-click to edit):*



## ✅ Check your understanding

Answer in your head (or out loud!) first, *then* open the answer.

**Q1. Why must unit conversions happen *before* outlier detection?**

<details>
<summary>Show answer</summary>

Because a systematic unit mix-up (90× prices) would otherwise be treated as 60 random 'outliers' and discarded, throwing away valid information.

</details>

**Q2. Name three ways to handle missing data and one risk of each.**

<details>
<summary>Show answer</summary>

**Drop**: loses data and can bias if gaps aren't random. **Fill with a global mean**: ignores structure (beans ≠ maize). **Group median/interpolation**: assumes the neighbourhood is representative (may smooth real spikes). Always add a `was_missing` flag.

</details>

**Q3. What's the difference between `pd.to_datetime(x)` and `pd.to_datetime(x, errors='coerce')`?**

<details>
<summary>Show answer</summary>

The default raises an error on an unparsable value; `coerce` converts it to `NaT` so you can inspect/handle bad dates.

</details>

**Q4. Why sort by date before `interpolate`, `rolling` or `shift`?**

<details>
<summary>Show answer</summary>

These operate on row order. If rows are shuffled, 'the previous row' isn't 'last week', and results are silently wrong.

</details>

**Q5. What does `groupby(...).transform('median')` return that `groupby(...).median()` doesn't?**

<details>
<summary>Show answer</summary>

`transform` returns a result **aligned to the original rows** (same length as the table), so you can fill, divide or compare row by row; `groupby().median()` returns one row per group.

</details>

## 🧾 Cheat-sheet

| I want to… | Code |
|---|---|
| Audit | `first_look(df)`, `df.isna().sum()`, `df.duplicated().sum()`, `df["c"].value_counts()` |
| Drop duplicates | `df.drop_duplicates(subset="id")` |
| Clean text | `.str.strip().str.lower().str.replace("-", " ", regex=False).str.title()` |
| Flag impossible values | `df[(df.price <= 0) \| (df.vol < 0)]` |
| Compare to group typical | `df.price / df.groupby("g")["price"].transform("median")` |
| Missing: drop / fill / interpolate | `dropna(subset=[..])`, `fillna(v)`, `interpolate()` |
| Parse dates | `pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")` |
| Date parts | `s.dt.year`, `.dt.month`, `.dt.quarter`, `.dt.day_name()`, `.dt.isocalendar().week` |
| Time series | `set_index("date").sort_index()`, `.resample("ME").mean()`, `.rolling(4).mean()`, `.shift(1)`, `.pct_change()` |
| Recode / bin | `s.map(dict)`, `np.select(conds, choices, default)`, `pd.cut(s, bins, labels)` |
| Guard rails | `assert df["id"].is_unique` |

## ➡️ Next up
**Lesson 5; Grouping, aggregating and merging.** Now that the table is clean, we'll ask *business* questions: *Which county is cheapest? Do high-production counties have lower prices?*